# Keep an unloaded field unloaded, even when it has a default

A list query loaded only the order identifier. The currency has a default and the code is made from the identifier, but neither was loaded: the partial instance holds neither, reads neither, dumps neither and runs no code for them.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, FieldNotLoadedError
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [2]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@meta(description="Stores the shop entities", domain=ShopDomain)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""


@entity(description="Customer order", resource=ShopStorage)
class OrderCoreEntity(BaseCoreEntity):
    """Describe an order whose fields have a default and a default factory."""

    id: str = Field(description="Order identifier")
    currency: str = Field(default="RUB", description="Currency code")
    code: str = Field(default_factory=lambda data: data["id"].upper(), description="Order code made from the identifier")


## Read and dump the partial order

In [3]:
order = OrderCoreEntity.partial(id="ord-4")
print("currency loaded:", order.is_field_loaded("currency"))
try:
    print(order.currency)
except FieldNotLoadedError as exc:
    print("reading currency:", type(exc).__name__)
else:
    raise AssertionError("An unloaded currency must not be readable")
print("Python dump:", order.model_dump())
print("JSON dump:", order.model_dump(mode="json"))


currency loaded: False
reading currency: FieldNotLoadedError
Python dump: {'id': 'ord-4'}
JSON dump: {'id': 'ord-4'}


## Load nothing at all

In [4]:
nothing = OrderCoreEntity.partial()
print("nothing loaded:", nothing.model_dump())


nothing loaded: {}


## Compare with a full order

In [5]:
full = OrderCoreEntity(id="ord-4")
print("full entity:", full.model_dump())


full entity: {'id': 'ord-4', 'currency': 'RUB', 'code': 'ORD-4'}


A partial entity holds only the fields it was given. A default or a default factory belongs to validation, which a full entity goes through and a partial one does not, so the factory that needs the identifier never runs for `partial()`.